In [1]:
import os
os.environ["PGPASSWORD"] = "aa8940aa"

import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pandas as pd
from sqlalchemy import text

from db.connection import DatabaseConnection

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["figure.dpi"] = 120
matplotlib.font_manager.fontManager.addfont(
    r"C:\Windows\Fonts\YuGothM.ttc"
)
plt.rcParams["font.family"] = "Yu Gothic"

def get_engine():
    return DatabaseConnection().get_engine()

print("Setup complete")

Setup complete


In [2]:
print("## Hold-out 最終評価")
print("テスト期間: 2022-01-01 〜 2024-12-31 (3年間)")
print("")
print("このノートブックは1回のみ実行すること。")
print("複数回実行すると Hold-out データへのリークが発生する。")
print("")

try:
    engine = get_engine()
    print("DB接続成功 — データをロード中...")
    # Note: 実際のロードは BacktestEngine.run() が行う
except Exception as e:
    print(f"DB接続失敗: {e}")
    print("実行には PostgreSQL 接続が必要です。")

## Hold-out 最終評価
テスト期間: 2022-01-01 〜 2024-12-31 (3年間)

このノートブックは1回のみ実行すること。
複数回実行すると Hold-out データへのリークが発生する。

DB接続成功 — データをロード中...


In [3]:
## バックテスト実行

from pipelines.training_pipeline import TrainingPipelineV5
from backtest.engine import BacktestEngine

print("モデル学習中（時間がかかります）...")
pipeline = TrainingPipelineV5()
models = pipeline.run("20150101", "20211231")

print("\nバックテスト実行中...")
engine = BacktestEngine(models, initial_bankroll=100000)
result = engine.run("20220101", "20241231")

print(f"\n=== バックテスト結果 ===")
print(f"  Total bets: {result.total_bets}")
print(f"  ROI: {result.total_roi:.1%}")
print(f"  Max DD: {result.max_drawdown:.1%}")
print(f"  Final bankroll: {result.final_bankroll:,.0f}")

C:\Users\hirom\AppData\Local\mise\installs\python\3.11.15\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


モデル学習中（時間がかかります）...


SubModel 'dirt' に学習データなし
SubModel 'turf' に学習データなし
2026/03/27 21:12:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/27 21:12:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/27 21:12:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/27 21:12:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/27 21:12:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/27 21:12:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/27 21:12:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/27 21:12:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/27 21:12:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `na


バックテスト実行中...

=== バックテスト結果 ===
  Total bets: 2766
  ROI: 63.8%
  Max DD: 100.0%
  Final bankroll: 0


In [4]:
print("""
## §13.2 合格基準 (v5.0)

| 基準 | 閾値 | 必須 |
|------|------|------|
| 複勝回収率 | >= 100% | YES |
| ワイド回収率 | >= 103% | YES |
| 全体回収率 | >= 101% | YES |
| 最大ドローダウン | <= 16% | YES |
| 月次100%超 | >= 22/36ヶ月 | YES |

※ 全て満たすことが本運用移行の前提条件。
""")


## §13.2 合格基準 (v5.0)

| 基準 | 閾値 | 必須 |
|------|------|------|
| 複勝回収率 | >= 100% | YES |
| ワイド回収率 | >= 103% | YES |
| 全体回収率 | >= 101% | YES |
| 最大ドローダウン | <= 16% | YES |
| 月次100%超 | >= 22/36ヶ月 | YES |

※ 全て満たすことが本運用移行の前提条件。



In [5]:
print("""
## §13.2 追加合格条件 (v5.1)

| 基準 | 閾値 | 必須 |
|------|------|------|
| EV補正モデルのMAE改善 | >= 10% | YES |
| 中穴ゾーンEV誤差改善 | >= 15% | YES |
| log_error SHAP寄与度 | > 0 | YES |

EV補正の有効性を定量的に確認する。
""")


## §13.2 追加合格条件 (v5.1)

| 基準 | 閾値 | 必須 |
|------|------|------|
| EV補正モデルのMAE改善 | >= 10% | YES |
| 中穴ゾーンEV誤差改善 | >= 15% | YES |
| log_error SHAP寄与度 | > 0 | YES |

EV補正の有効性を定量的に確認する。



In [6]:
print("""
## §13.2 追加合格条件 (v5.4)

| 基準 | 閾値 | 必須 |
|------|------|------|
| P補正AUC改善 | >= 1% | YES |
| P/E補正相関 | < 0.3 | YES |
| E補正MAE改善 (winner) | 改善 | YES |
| Wide Var_proxy 正確性 | EV/(E×sqrt(P)) 一致 | YES |

P/E分解補正の有効性を確認する。
""")


## §13.2 追加合格条件 (v5.4)

| 基準 | 閾値 | 必須 |
|------|------|------|
| P補正AUC改善 | >= 1% | YES |
| P/E補正相関 | < 0.3 | YES |
| E補正MAE改善 (winner) | 改善 | YES |
| Wide Var_proxy 正確性 | EV/(E×sqrt(P)) 一致 | YES |

P/E分解補正の有効性を確認する。



In [7]:
print("""
## 月次ROI推移

36ヶ月の月次ROIを棒グラフで表示:
  - 緑色: ROI >= 100% (黒字月)
  - 赤色: ROI < 100% (赤字月)
  - 22ヶ月以上が緑色であること

実装:
  result.monthly_returns を DataFrame に変換
  plt.bar(months, roi_values, color=conditions)
  plt.axhline(y=1.0, color='black', linestyle='--', alpha=0.5)
""")


## 月次ROI推移

36ヶ月の月次ROIを棒グラフで表示:
  - 緑色: ROI >= 100% (黒字月)
  - 赤色: ROI < 100% (赤字月)
  - 22ヶ月以上が緑色であること

実装:
  result.monthly_returns を DataFrame に変換
  plt.bar(months, roi_values, color=conditions)
  plt.axhline(y=1.0, color='black', linestyle='--', alpha=0.5)



In [8]:
print("""
## Bankroll 遷移

全期間の bankroll 推移を折れ線グラフで表示:
  - 初期資金: 100,000円
  - max drawdown 領域を赤で塗りつぶし
  - RecoveryState (NORMAL/REDUCED/RECOVERING) を背景色で表示

実装:
  bet_history から cumsum で bankroll 遷移を計算
  plt.fill_between で DD 領域を描画
""")


## Bankroll 遷移

全期間の bankroll 推移を折れ線グラフで表示:
  - 初期資金: 100,000円
  - max drawdown 領域を赤で塗りつぶし
  - RecoveryState (NORMAL/REDUCED/RECOVERING) を背景色で表示

実装:
  bet_history から cumsum で bankroll 遷移を計算
  plt.fill_between で DD 領域を描画



In [9]:
print("""
## 最終判定

全合格基準を満たした場合:
  → 「本運用への移行可」が出力される
  → 500円/bet での小額実運用を開始

合格基準を満たさなかった場合:
  → 不合格の項目を特定
  → モデル/特徴量/パラメータの調整が必要
  → TrainingPipelineV5 で再学習

⚠️ 重要: Hold-out 期間は 1回のみ使用すること。
   複数回の実行は Hold-out の意味を失う。
""")


## 最終判定

全合格基準を満たした場合:
  → 「本運用への移行可」が出力される
  → 500円/bet での小額実運用を開始

合格基準を満たさなかった場合:
  → 不合格の項目を特定
  → モデル/特徴量/パラメータの調整が必要
  → TrainingPipelineV5 で再学習

⚠️ 重要: Hold-out 期間は 1回のみ使用すること。
   複数回の実行は Hold-out の意味を失う。

